# v9b Normative JEPA + Conformal training on Google Colab

**Pipeline** (3 stages, all crash-safe, all use --resume auto):
1. **Stage 1: JEPA self-supervised pretrain on healthy MRI** (~12 hr T4 / ~6 hr A100)
2. **Stage 2: train DDPM counterfactual decoder + SDF geometric tower** (~6 hr T4 / ~3 hr A100)
3. **Stage 3: calibrate conformal threshold + run inference** (CPU OK)

**Before you start**: upload `dataset_v8.zip` and `colab_bundle.zip` to `MyDrive/neurolens/`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Extract dataset + code

In [ ]:
import os, shutil, time
DRIVE='/content/drive/MyDrive/neurolens'
LOCAL='/content/neurolens'
os.makedirs(LOCAL, exist_ok=True)
if not os.path.isdir(f'{LOCAL}/dataset_v8/train/images'):
    t0=time.time()
    !unzip -q '{DRIVE}/dataset_v8.zip' -d {LOCAL}/
    # Fix PowerShell backslash paths if needed
    from pathlib import Path
    for p in list(Path(LOCAL).iterdir()):
        if '\\' in p.name:
            new=Path(LOCAL)/p.name.replace('\\','/'); new.parent.mkdir(parents=True,exist_ok=True)
            shutil.move(str(p), str(new))
    # Promote train/val/test under dataset_v8/ if extracted flat
    os.makedirs(f'{LOCAL}/dataset_v8', exist_ok=True)
    for s in ['train','val','test']:
        src=f'{LOCAL}/{s}'; dst=f'{LOCAL}/dataset_v8/{s}'
        if os.path.isdir(src) and not os.path.isdir(dst): shutil.move(src,dst)
    print(f'extracted in {time.time()-t0:.0f}s')
!unzip -o '{DRIVE}/colab_bundle.zip' -d {LOCAL}/
!ls {LOCAL}/src/

In [ ]:
!pip install -q segmentation-models-pytorch timm scikit-image
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. STAGE 1 — I-JEPA pretrain (healthy brains only)

Saves checkpoints to Drive: `MyDrive/neurolens/v9b_jepa/`. Auto-resumes on disconnect.

In [ ]:
JEPA_OUT='/content/drive/MyDrive/neurolens/v9b_jepa'
!cd /content/neurolens && python src/train_v9b_stage1_jepa.py \
    --data_dir /content/neurolens/dataset_v8 \
    --output_dir '{JEPA_OUT}' \
    --epochs 200 --batch_size 64 --image_size 256 \
    --embed_dim 384 --depth 12 --heads 6 \
    --num_workers 4 --amp --cache_in_ram \
    --checkpoint_every_steps 200 --resume auto

## 3. STAGE 2 — DDPM counterfactual decoder + SDF geometric tower

Uses the pretrained JEPA from Stage 1 (frozen) as healthy-latent feature extractor.

In [ ]:
S2_OUT='/content/drive/MyDrive/neurolens/v9b_stage2'
JEPA_CKPT=f'{JEPA_OUT}/last.pt'
!cd /content/neurolens && python src/train_v9b_stage2.py \
    --data_dir /content/neurolens/dataset_v8 \
    --output_dir '{S2_OUT}' --jepa_ckpt '{JEPA_CKPT}' \
    --epochs 100 --batch_size 32 --image_size 256 \
    --num_workers 4 --amp --cache_in_ram \
    --checkpoint_every_steps 200 --resume auto

## 4. STAGE 3 — Calibrate conformal threshold on healthy held-out set

In [ ]:
import sys, torch, numpy as np
sys.path.insert(0, '/content/neurolens')
from src.research.v9b_model import V9BModel
from src.research.jepa_conformal import JepaConformalCalibrator
from src.train_v9b_stage1_jepa import HealthyOnlyDataset
from torch.utils.data import DataLoader

m = V9BModel.from_checkpoints(f'{JEPA_OUT}/last.pt', f'{S2_OUT}/last.pt',
                                conformal_json=None, image_size=256, device='cuda')
calib_ds = HealthyOnlyDataset('/content/neurolens/dataset_v8', image_size=256)
calib_loader = DataLoader(calib_ds, batch_size=8, shuffle=False, num_workers=2)
scores = []
with torch.no_grad():
    for x in calib_loader:
        x = x.cuda()
        emap = m.jepa.prediction_error_map(x)  # (B,1,H,W)
        per_scan = emap.flatten(1).quantile(0.95, dim=1)  # 95th pct per scan
        scores.extend(per_scan.cpu().tolist())
cal = JepaConformalCalibrator(alpha=0.10)
rep = cal.calibrate(scores, verbose=True)
CONF='/content/drive/MyDrive/neurolens/v9b_conformal.json'
cal.save(CONF); print(f'saved conformal to {CONF}')

## 5. Inference on a test image (end-to-end)

In [ ]:
TEST_IMG='/content/neurolens/dataset_v8/test/images/' + sorted(__import__('os').listdir('/content/neurolens/dataset_v8/test/images'))[0]
OUT='/content/v9b_inference_out'
!python /content/neurolens/src/v9b_inference.py \
    --jepa_ckpt '{JEPA_OUT}/last.pt' --stage2_ckpt '{S2_OUT}/last.pt' \
    --conformal '{CONF}' --image '{TEST_IMG}' --output_dir '{OUT}' \
    --combine_mode weighted_sum --ddpm_steps 50
!ls {OUT}

## Troubleshooting

- **Colab disconnects**: re-run the relevant Stage cell. `--resume auto` picks up from `last.pt` on Drive.
- **OOM at batch 64**: reduce `--batch_size 32` or `--batch_size 16`.
- **JEPA loss not decreasing**: check `--lr 1.5e-4` (the default); may need lower (1e-4) if loss is unstable.
- **DDPM samples are noise**: needs ~50+ epochs in Stage 2 before counterfactuals look reasonable.
- **Few healthy scans**: dataset_v8 has ~1.5K Kaggle no_tumor. For best JEPA, mount IXI/OASIS via `--extra_dirs`.
- **Out-of-the-box test**: tests/test_v9b_components.py runs 18 unit tests in <30s, validates every module.